# Fix get_history

In [ ]:
from datetime import datetime, timedelta, date

## Using nsepython - Equity

In [ ]:
from nsepython import equity_history, index_history, derivative_history, expiry_history

In [ ]:
eq_symbol = 'SBIN'
series = 'EQ'
days = 41
end = datetime.now().date()
start = end-timedelta(days=days)

In [ ]:
# convert datetimes to text for equities
eq_end = end.strftime('%d-%m-%Y')
eq_start = start.strftime('%d-%m-%Y')

eq_hist = equity_history(eq_symbol, series, eq_start, eq_end)
eq_hist.head()

In [28]:
from nsepython import nse_quote_ltp
# nse_quote_ltp(symbol='NIFTY BANK', strikePrice=42100)
print(nse_quote_ltp("BANKNIFTY","10-Jun-2021","PE",42100))

UnboundLocalError: local variable 'lastPrice' referenced before assignment

## Using nsepython - Index

In [ ]:
idx_symbol = "NIFTY BANK"
start_date = start.strftime('%d-%b-%Y')
end_date = end.strftime('%d-%b-%Y')

idx_hist = index_history(idx_symbol, start_date, end_date)
idx_hist.head()

# Custom codes to make things work

In [ ]:
import requests
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:74.0) Gecko/20100101 Firefox/74.0'
}

def nse_json(url: str):
    """Fetch json from nse for the url provided"""
    try:
        output = requests.get(url,headers=headers).json()

    except ValueError:
        try:
            s=requests.Session()
            output = s.get("http://nseindia.com",headers=headers)
            output = s.get(url,headers=headers).json()

        except ValueError: # for csv loads generating JSONDecodeError
            output = requests.get(url).text
        
    return output

## Equity - works!

In [ ]:
url = 'https://www.nseindia.com/api/historical/cm/equity?symbol=SBIN&series=[%22EQ%22]&from=22-02-2023&to=03-04-2023'
nse_json(url)

## Derivatives

In [ ]:
symbol = "SBIN"
start_date = "15-05-2021"
end_date ="15-06-2021"
instrumentType = "OPTSTK"
expiry_date ="24-Jun-2021"
strikePrice = 300
optionType="PE"

In [ ]:
strikePrice = "%.2f" % strikePrice
strikePrice = str(strikePrice)

In [ ]:
strikePrice

In [ ]:
nsefetch_url = "https://www.nseindia.com/api/historical/fo/derivatives?&from="+ \
                str(start_date)+"&to="+str(end_date)+"&optionType="+ \
                    optionType+"&strikePrice="+ \
                        str(strikePrice)+"&expiryDate="+ \
                            expiry_date+"&instrumentType="+ \
                                instrumentType+"&symbol="+symbol+""

In [ ]:
nse_json(nsefetch_url)

## Indexes

In [ ]:
url = 'https://niftyindices.com/Backpage.aspx/getpepbHistoricaldataDBtoString'
data = {'name':'NIFTY','startDate':'22-02-2023','endDate':'03-04-2023'}
idx_header = {'Connection': 'keep-alive', 
'sec-ch-ua': '" Not;A Brand";v="99", "Google Chrome";v="91", "Chromium";v="91"', 
'Accept': 'application/json, text/javascript, */*; q=0.01', 
'DNT': '1', 
'X-Requested-With': 'XMLHttpRequest', 
'sec-ch-ua-mobile': '?0', 
'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.77 Safari/537.36',
'Content-Type': 'application/json; charset=UTF-8', 
'Origin': 'https://niftyindices.com', 
'Sec-Fetch-Site': 'same-origin', 
'Sec-Fetch-Mode': 'cors', 
'Sec-Fetch-Dest': 'empty', 
'Referer': 'https://niftyindices.com/reports/historical-data'}

In [ ]:
r = requests.post(url, data=data, headers=idx_header)

In [ ]:
def nsefetch(payload):
    try:
        output = requests.get(payload,headers=headers).json()
        #print(output)
    except ValueError:
        s =requests.Session()
        output = s.get("http://nseindia.com",headers=headers)
        output = s.get(payload,headers=headers).json()
    return output


# Does not work!!!

## Using nsepython - Derivative

In [ ]:
symbol = "SBIN"
start_date = start.strftime('%d-%m-%Y')
end_date = end.strftime('%d-%m-%Y')
instrumentType = "options"
expiry_date =datetime.strptime(expiry_history(symbol)[2], '%d-%b-%Y').strftime('%d-%m-%Y')
strikePrice = 300
optionType="PE"

In [ ]:
derivative_history(symbol,start_date,end_date,instrumentType,expiry_date,strikePrice,optionType)

In [ ]:
symbol = "SBIN"
start_date = "15-05-2021"
end_date ="15-06-2021"
instrumentType = "options"
expiry_date ="24-Jun-2021"
strikePrice = 300
optionType="PE"
print(derivative_history(symbol,start_date,end_date,instrumentType,expiry_date,strikePrice,optionType))

In [ ]:
url = 'https://www.nseindia.com/api/historical/cm/equity?symbol=SBIN&series=[%22'+series+'%22]&from=03-04-2022&to=13-05-2022'

In [ ]:
nse_json(url)

In [ ]:
headers = {
    'Connection': 'keep-alive',
    'Cache-Control': 'max-age=0',
    'DNT': '1',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/79.0.3945.79 Safari/537.36',
    'Sec-Fetch-User': '?1',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-Mode': 'navigate',
    'Accept-Encoding': 'gzip, deflate, br',
    'Accept-Language': 'en-US,en;q=0.9,hi;q=0.8',
}

In [ ]:
import pandas as pd
def equity_history_virgin(symbol,series,start_date,end_date):
    url="https://www.nseindia.com/api/historical/cm/equity?symbol="+symbol+"&series=[%22"+series+"%22]&from="+str(start_date)+"&to="+str(end_date)+""
    payload = nsefetch(url)
    # return pd.DataFrame.from_records(payload["data"])
    return payload

In [ ]:
def derivative_history_virgin(symbol,start_date,end_date,instrumentType,expiry_date,strikePrice="",optionType=""):

    instrumentType = instrumentType.lower()

    if(instrumentType=="options"):
        if("NIFTY" in symbol): instrumentType="FUTSTK"
        instrumentType="OPTSTK"
    if(instrumentType=="futures"):
        if("NIFTY" in symbol): instrumentType="OPTIDX"
        instrumentType="FUTIDX"

    if(((instrumentType=="OPTIDX")or (instrumentType=="OPTSTK")) and (expiry_date!="")):
        strikePrice = "%.2f" % strikePrice
        strikePrice = str(strikePrice)

    nsefetch_url = "https://www.nseindia.com/api/historical/fo/derivatives?&from="+str(start_date)+"&to="+str(end_date)+"&optionType="+optionType+"&strikePrice="+strikePrice+"&expiryDate="+expiry_date+"&instrumentType="+instrumentType+"&symbol="+symbol+""
    payload = nsefetch(nsefetch_url)
    logging.info(payload)
    return pd.DataFrame.from_records(payload["data"])

# Experiments

In [ ]:
from nsepy import get_history
# Stock options (Similarly for index options, set index = True)
stock_opt = get_history(symbol="SBIN",
                        start=date(2015,1,1),
                        end=date(2015,1,10),
                        option_type="CE",
                        strike_price=300,
                        expiry_date=date(2015,1,29))